# **Feature Engineering**

## Objectives

* Split the data into train and test sets
* Build a reproducible preprocessing pipeline that implements the cleaning actions and feature transformations explored and validated in the[cleaning](/jupyter_notebooks/04_cleaning.ipynb) notebook as well as required encoding
* Fit the pipeline to on train data and transform both train and test sets 


## Inputs

* outputs/datasets/cleaned/HotelBookingsValid.csv

## Outputs

* Feature engineering pipeline saved as outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl
* X_train, X_test, y_train and y_test to outputs/ml_pipeline/preprocessing as .csv

## Additional Comments

* ⚠️ TBC ⚠️


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

# Load Data

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsValid.csv")
df.head(3)

---

## Split train and test set

In [ ]:
from sklearn.model_selection import train_test_split

df_data = df.drop("is_canceled", axis=1)
target = df["is_canceled"]

print(f"df_data shape: {df_data.shape}, target shape: {target.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    df_data, target, test_size=0.2, random_state=4, stratify=target
)

display_shapes = pd.Series({"X_train": X_train.shape,
              "X_test": X_test.shape,
              "y_train": y_train.shape,
              "y_test": y_test.shape}, name="split_shapes")
display_shapes

---

## Pipeline

**Pipeline actions**

|Feature|Meaning|Data type|Preprocessing actions|Prediction model actions|Experimental prediction model alternatives|Cluster model actions|Experimental cluster model alternatives|
|---|---|---|---|---|---|---|---|
|hotel|booking location|nominal|one-hot|||||
|is_canceled|booking was cancelled|binary|none|target only|SMOTE|exclude||
|lead_time|days before arrival booking made|numeric|winsorize||log/skew transformation; binning|scaling|log/skew transformation|
|arrival_date_year|year of arrival|numeric|drop|||||
|arrival_date_month|month of arrival|nominal|defer|one-hot|cyclical encoding|cyclical encoding||
|arrival_date_week_number|week of the year of arrival|numeric|none|exclude|cyclical encoding|exclude|cyclical encoding|
|arrival_date_day_of_month|day in the month of arrival|numeric|none|||drop||
|stays_in_weekend_nights|weekend nights stayed|numeric|winsorize||binning|scaling|binning|
|stays_in_week_nights|week nights stayed|numeric|winsorize||binning|scaling|binning|
|adults|number of adults on the booking|numeric|none||binning|scaling|binning|
|children|number of children|numeric|none||binary, binning|binary|binary combined (children + babies)|
|babies|number of babies|numeric|none||binary, binning|binary|binary combined (children + babies)|
|meal|meal plan booked|nominal|replace "Undefined" with "SC", one-hot|||||
|country|country of origin|nominal|impute missing data|ordinal-categorical|frequency, target|frequency|regional grouping, exclude|
|market_segment|demographic information|nominal|one-hot|||||
|distribution_channel|demographic information|nominal|one-hot|||||
|is_repeated_guest|if the guest has booked before|binary|none|||||
|previous_cancellations|how many times the guest has cancelled before|numeric|none||binary|binary|binning|
|previous_bookings_not_canceled|how many times the guest has completed a booking|numeric|none||log/skew transformation; binning|binary|binning|
|reserved_room_type|Room code booked|nominal|one-hot|||||
|deposit_type|Booking security policy|nominal|one-hot|||||
|agent|ID code of booking agent|nominal|impute missing data||frequency, target, exclude|binary|frequency|
|company|ID code of company the guest is travelling for|nominal|drop|||||
|days_in_waiting_list|How long the booking waited for confirmation|numeric|none||binary, skew transformation|binary|binning|
|customer_type|demographic information|nominal|one-hot|||||
|adr|cost per night of the booking|numeric|winsorize||log/skew transformation; binning|scaling|log/skew transformation; binning|
|required_car_parking_spaces|car parking spaces needed|numeric|none||binary|binary||
|total_of_special_requests|special requests made|numeric|none||binning, binary|binning|scaling, binary|

**Cleaning steps**
1. Drop `company` and `arrival_date_year`
2. Replace "Undefined" with "SC" in `meal`
3. Impute missing values
4. Outlier handling

* Create preprocessing pipeline

In [ ]:
print("X_train shape ", X_train.shape)

In [ ]:
from feature_engine.selection import DropFeatures

X_train_copy = X_train.copy()
drop_transformer = DropFeatures(features_to_drop=["company", "arrival_date_year"])
pipeline_step1 = drop_transformer.fit_transform(X_train_copy)
pipeline_step1.shape


In [ ]:
from sklearn.preprocessing import FunctionTransformer
from utils.custom_transformers import undefined_meal

replace_transformer = FunctionTransformer(undefined_meal)
pipeline_step2 = replace_transformer.fit_transform(pipeline_step1)
pipeline_step2["meal"].value_counts()

In [ ]:
pipeline_step2["agent"].describe()

In [ ]:
pipeline_step2["agent"].isnull().sum()

In [ ]:
from feature_engine.imputation import ArbitraryNumberImputer, CategoricalImputer

agent_imputer = ArbitraryNumberImputer(arbitrary_number=0, variables="agent")
pipeline_step3 = agent_imputer.fit_transform(pipeline_step2)
print("Missing values: ", pipeline_step3["agent"].isnull().sum())
pipeline_step3["agent"].describe()


In [ ]:
country_imputer = CategoricalImputer(imputation_method="frequent", variables="country")
pipeline_step4 = country_imputer.fit_transform(pipeline_step3)
print("Missing values: ", pipeline_step4["country"].isnull().sum())
pipeline_step4["country"].value_counts()

In [ ]:
pipeline_step4.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

outlier_cols = ["lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights"]

def plot_outliers(cols):
    ncols = 2
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 8))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.boxplot(data=pipeline_step4,
                     x=col,
                     ax=axs[i])   

    plt.tight_layout()

plot_outliers(outlier_cols)

In [ ]:
from feature_engine.outliers import Winsorizer

winsorizer = Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=outlier_cols)
pipeline_step5 = winsorizer.fit_transform(pipeline_step4)
pipeline_step5.describe()

In [ ]:
def plot_outliers(cols):
    ncols = 2
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 8))
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.boxplot(data=pipeline_step5,
                     x=col,
                     ax=axs[i])   

    plt.tight_layout()

plot_outliers(outlier_cols)

**Feature Engineering**
1. One-hot encode categorical variables

In [ ]:
categorical_cols = ["hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type", "deposit_type", "customer_type"]
for col in categorical_cols:
    print(f"{col} has {df[col].nunique()} categories")

In [ ]:
from feature_engine.encoding import OneHotEncoder

encoder = OneHotEncoder(variables=categorical_cols, drop_last=True)
pipeline_step6 = encoder.fit_transform(pipeline_step5)
pipeline_step6.head(3)

In [ ]:
from sklearn.pipeline import Pipeline

def preprocessing_pipeline():

    pipeline_base = Pipeline([
        ("DropFeatures", DropFeatures(features_to_drop=["company", "arrival_date_year"])),
        ("FunctionTransformer", FunctionTransformer(undefined_meal)),
        ("ArbitraryNumberImputer", ArbitraryNumberImputer(arbitrary_number=0, variables="agent")),
        ("CategoricalImputer", CategoricalImputer(imputation_method="frequent", variables="country")),
        ("Winsorizer", Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=outlier_cols)),
        ("OneHotEncoder", OneHotEncoder(variables=categorical_cols, drop_last=True))
    ])

    return pipeline_base

---

## Save Files

In [ ]:
import os
try:
  os.makedirs(name='outputs/ml_pipeline/preprocessing')
except Exception as e:
  print(e)


**Train Set**

In [ ]:
X_train.to_csv("outputs/ml_pipeline/preprocessing/X_train.csv", index=False)
y_train.to_csv("outputs/ml_pipeline/preprocessing/y_train.csv", index=False)

**Test Set**

In [ ]:
X_test.to_csv("outputs/ml_pipeline/preprocessing/X_test.csv", index=False)
y_test.to_csv("outputs/ml_pipeline/preprocessing/y_test.csv", index=False)

**Pipeline**

In [ ]:
pipeline_preprocessing = preprocessing_pipeline()
pipeline_preprocessing

In [ ]:
import joblib

pipeline_preprocessing = preprocessing_pipeline()

joblib.dump(value=pipeline_preprocessing,
            filename="outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl")